In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")


# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti 
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( puti )
importlib.reload( uti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)
importlib.reload(LL)


In [ ]:
from myPythonTools.YAML import get_exp_list as Xl
importlib.reload(Xl)


In [ ]:
print( sys.path )
############################
# Initialize dict for regridding
############################
RgObs={}


In [ ]:
### Make a list of files
import yaml
#from box import Box


with open('../YAML/configure_plotting.yaml', 'r') as file:
    cfgPlot = yaml.safe_load(file)


######################################################
season = 'jja' # cfgPlot['latlon']['season']
valData  = cfgPlot['latlon']['validation']['data']
valYears = cfgPlot['latlon']['validation']['years']
print( f"Season= {season}" )
print( valData , valYears )

# other ... 
#fld='TAUX'
#fld='PRECT'
fld='PREC_CONV_FRAC'
#fld='ICEFRAC'
#fld='U'
#fld='UTGWSPEC'
#fld='UTGW_MOVMTN'
#fld='RESTOM'
#fld='U10'
#fld='TS'
#fld='LHFLX'
#fld='SWCF'
#fld='LWCF'
#fld='CLDICE'
#fld='WSP'
#fld='WP2_CLUBB'
#fld='SHFLX'
plot_validation=False #True
use_desc_in_title = True #True
use_ymdPat_from_yaml = True

override_ymdPat = False

if (override_ymdPat==True):
    use_ymdPat_from_yaml = False
    ymdPat_spec=[20,21]


In [ ]:

#exp_file,xl= Xl.fGW_tests() 


#exp_file='GW_Xtreme_ppe.yaml'
#xl= ['control','nofgw','NOogw','NOgwATall',] #'ne240x2_QxQsst']
#exp_file='HiResRuns.yaml'
#xl= ['control_c64116','ne120_flt_c64116','ne240_c63153_2004',] #'ne240x2_QxQsst']
#exp_file='Tiedtke_tests.yaml'
#xl=['tiedtke3K']
#xl= ['B1850_x238',] #'ne240x2_QxQsst']

#exp_file,xl= Xl.tausrfRuns() 
#exp_file,xl= Xl.fGW_tests() 
exp_file,xl= Xl.Official() 
exp_file='HiResRuns.yaml'
xl=['cam_hr_HR','cam_hr_LR']
exp_file='fGW_tests.yaml'
xl=['CTL_c64144']


with open(f'../YAML/{exp_file}', 'r') as file:
    cfg = yaml.safe_load(file)


In [ ]:

#exp_file,xl= Xl.Official()  #Utility()

#with open(f'../YAML/{exp_file}', 'r') as file:
#    cfg = yaml.safe_load(file)

Al =[]
for x in xl:
    exp, subd, Src, Hkey, Dst, useri , hsPat = cfg[x]['name'] , cfg[x]['subdir'] , cfg[x]['SrcGrid'] , cfg[x]['Hkey'] , cfg[x]['DstGrid'] , cfg[x]['user'] , cfg[x]['hsPat'] 




    if (use_ymdPat_from_yaml==True):
        ymdPat=cfg[x]['ymdPat']
        if ( 'ymdPat_is_range' in cfg[x] ):
            ymdPat_is_range = cfg[x]['ymdPat_is_range']
        else:
            ymdPat_is_range = False
    elif (override_ymdPat==True):
        ymdPat=ymdPat_spec
        ymdPat_is_range = True
    else:
        ymdPat = '*' # ['1985-*', '1986-*']
        ymdPat_is_range = False

    print( exp, subd, Src, Hkey, Dst, useri )
    print( ymdPat  )
    print( ymdPat_is_range )
    
    A = uti.MakeDict4Exp( exp=exp , user=useri, subd=subd , 
                     hsPat=hsPat , ymdPat=ymdPat,
                     ymdPat_is_range =ymdPat_is_range , 
                     verbose=True, open_dataset=True )
    
    A['Src']=Src
    A['Hkey']=Hkey
    if 'desc' in cfg[x]:
        A['desc'] = cfg[x]['desc']
    if 'gw_parameters' in cfg[x]:
        A['gw_parameters'] = cfg[x]['gw_parameters']
    Al.append( A )
    print( f'{A.exp} Years {len(A.X.time) /12. } \n')


In [ ]:
f='/glade/derecho/scratch/hannay/archive/b.e30_alpha08o.B1850C_MTso.ne30_t232_wgx3.338/atm/hist/b.e30_alpha08o.B1850C_MTso.ne30_t232_wgx3.338.cam.h0a.0018-12.nc'
Poo=xr.open_dataset(f)
latpoo=Poo.lat.values




In [ ]:
fsnt_c=Al[0].X.FSNT.values
flnt_c=Al[0].X.FLNT.values
area_c=Al[0].X.area.values


In [ ]:
flnt_c.shape
nt=flnt_c.shape[0]
restom_c=np.zeros(nt)
for t in np.arange( nt ):
    restom_c[t] = np.sum( ( fsnt_c[t,:] - flnt_c[t,:] )*area_c[:] ) /np.sum( area_c[:] )  

In [ ]:
plt.plot( restom_c )

In [ ]:
importlib.reload( GrU )

In [ ]:
Src='ne30pg3'
Dst = 'latlon_weird_192x288'
DstInfo = GrU.gridInfo(Dst) #,Vgrid=DstVgrid)
lat_a,lon_a = GrU.latlon( scrip= DstInfo['scrip'], Hkey=DstInfo['Hkey'] )
lev_a=A.X.lev.values
zlev_a=-7.0*np.log( lev_a/1_000. )

RegridObj = GrU.regrid_object_lib(RgOb=RgObs, src=Src, dst=Dst, RegridMethod='BILINEAR')


#AAa_s_yx=RgF.Horz(xfld_Src=AAa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
#A['AA_s_yx'] = AAa_s_yx


In [ ]:
fsnt_yx=RgF.Horz(xfld_Src=fsnt_c , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
flnt_yx=RgF.Horz(xfld_Src=flnt_c , Src=Src, Dst=Dst , RegridObj_In= RegridObj )

#area_yx = np.cos(np.deg2rad(lat_a))

In [ ]:
nx=len(lon_a)
area_yx=np.tile( wgt_y[:,None] , (1,nx) )
plt.contour( area_yx)

In [ ]:
restom_yx=np.zeros(nt)
for t in np.arange( nt ):
    restom_yx[t] = np.sum( ( fsnt_yx[t,:] - flnt_yx[t,:] )*area_yx[:] , axis=(0,1) ) /np.sum( area_yx[:,:] , axis=(0,1))  

In [ ]:
#plt.plot( restom_c )
fig,axs=plt.subplots( 1,2, figsize=( 16,8) )
axs[0].plot( restom_c, label='native' )
axs[0].plot( restom_yx , linestyle=':',label='from bilinear remap')
axs[0].legend()
axs[1].plot( restom_yx-restom_c )
print( ( restom_yx-restom_c ).mean())

In [ ]:

am=Al[1].X.hyam.values
bm=Al[1].X.hybm.values
ai=Al[1].X.hyai.values
bi=Al[1].X.hybi.values
ps=Al[1].X.PS.values




pmid,pint,delp = MkP.Pressure ( am=am, bm=bm , ai=ai, bi=bi, ps=ps , p_00=100_000., Gridkey='tzc' )

cldliq=Al[1].X.CLDLIQ.values
nt,nz,ncol=np.shape( cldliq )
lwp2=np.zeros( (nt, ncol ) )

print( np.shape( lwp2 ) )

for m in np.arange( 12 ):
    lwp2[m,:] = np.sum( delp[m,:,:] * cldliq[m,:,:] , axis=0 )


In [ ]:
lwp1=Al[1].X.TGCLDLWP.values


In [ ]:
print( np.shape(lwp1) , np.shape(lwp2) )

plt.scatter( 9.8*lwp1[0,:],lwp2[0,:] )





In [ ]:
plt.tricontourf( lonc,latc, 9.8*lwp1[0,:]-lwp2[0,:] )

In [ ]:
#plt.plot( np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[1].AA_s_yx, axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[2].AA_s_yx, axis=1 ) )


In [ ]:
#plt.plot( np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*(Al[1].AA_s_yx -Al[0].AA_s_yx) , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*(Al[2].AA_s_yx -Al[0].AA_s_yx), axis=1 ) )


In [ ]:
#####